# <b>The RL Agent

Installing libraries

## Importing Libraries

In [1]:
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
device = 'cuda'

## Import the Dataset

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
dataset_path = "/content/drive/MyDrive/Kaarshika/datasets/kaarshika_reward_model_5000.jsonl"

In [5]:
dataset = load_dataset('json', data_files = dataset_path)

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'chosen', 'rejected'],
        num_rows: 5000
    })
})


In [7]:
dataset = dataset.shuffle(seed = 42)

In [8]:
dataset['train'][0]

{'id': 'KAARSHIKA_RM_01536',
 'prompt': 'Crop: maize\nStage: flowering\nSoil moisture: high\nRain probability: low\nTemperature: high\nHumidity: low\nWater availability: limited\n\nWhich action is more appropriate for this farm state?',
 'chosen': 'IMPROVE_DRAINAGE',
 'rejected': 'MONITOR'}

### Train-Test Split

In [9]:
dataset_split = dataset['train'].train_test_split(test_size = 0.05, seed = 42)
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

dataset_split

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'chosen', 'rejected'],
        num_rows: 4750
    })
    test: Dataset({
        features: ['id', 'prompt', 'chosen', 'rejected'],
        num_rows: 250
    })
})

## Model & Tokenizer setup

In [10]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 1)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Create Prompt-Chosen & Prompt-Rejected

In [11]:
def add_combined_columns(example):
    example["prompt_chosen"] = (
        example["prompt"] + "\nAction: " + example["chosen"]
    )

    example["prompt_rejected"] = (
        example["prompt"] + "\nAction: " + example["rejected"]
    )

    return example

In [12]:
train_dataset = train_dataset.map(add_combined_columns)
test_dataset = test_dataset.map(add_combined_columns)

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

## Tokenization

In [13]:
max_length = 512

In [14]:
def preprocess_function(examples):
    tokenized_chosen = tokenizer(examples['prompt_chosen'], truncation = True, max_length = max_length, padding = 'max_length')
    tokenized_rejected = tokenizer(examples['prompt_rejected'], truncation = True, max_length = max_length, padding = 'max_length')

    return {
        "input_ids_chosen": tokenized_chosen['input_ids'],
        "attention_mask_chosen": tokenized_chosen['attention_mask'],
        "input_ids_rejected": tokenized_rejected['input_ids'],
        "attention_mask_rejected": tokenized_rejected['attention_mask']
    }

In [15]:
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True
)

test_dataset = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

## LoRA Configuration

In [16]:
peft_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    inference_mode = False,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.1,
    target_modules = ['q_lin', 'v_lin'],
)

## Reward Config

In [17]:
from trl import RewardConfig

In [18]:
training_args = RewardConfig(
    output_dir="/content/drive/MyDrive/Kaarshika/models/reward_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=10,
    num_train_epochs = 2,
    eval_strategy = 'steps',
    eval_steps = 50,
    run_name="kaarshika-distilbert-reward-v1"
)

## RewardTrainer

In [19]:
from trl import RewardTrainer

In [20]:
trainer = RewardTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(


In [24]:
output_dir = '/content/drive/MyDrive/Kaarshika/models/reward_model'

In [25]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [26]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy
50,0.677900,0.666969,0.696000
100,0.646300,0.623940,0.720000
150,0.597300,0.558106,0.752000
200,0.573500,0.522215,0.760000
250,0.540400,0.513780,0.760000


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ [CLS] crop : wheat stage : harvesting soil    │ [CLS] crop : wheat stage : harvesting soil   │ [0.4991, 0.5009] │
│ moisture : high rain probability : moderate   │ moisture : high rain probability : moderate  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ harvest [SEP]                                 │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.4953, 0.5047] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.4625, 0.5375] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : moderate water  │ temperature : high humidity : moderate water │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ irrigate [SEP]                               │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : maize stage : vegetative soil    │ [CLS] crop : maize stage : vegetative soil   │ [0.4656, 0.5344] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : moderate humidity : high water  │ temperature : moderate humidity : high water │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ reduce _ irrigation [SEP]                    │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ [CLS] crop : wheat stage : harvesting soil    │ [CLS] crop : wheat stage : harvesting soil   │ [0.5814, 0.4186] │
│ moisture : high rain probability : moderate   │ moisture : high rain probability : moderate  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ harvest [SEP]                                 │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.4834, 0.5166] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.408, 0.592]   │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : moderate water  │ temperature : high humidity : moderate water │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ irrigate [SEP]                               │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : maize stage : vegetative soil    │ [CLS] crop : maize stage : vegetative soil   │ [0.5096, 0.4904] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : moderate humidity : high water  │ temperature : moderate humidity : high water │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ reduce _ irrigation [SEP]                    │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ [CLS] crop : wheat stage : harvesting soil    │ [CLS] crop : wheat stage : harvesting soil   │ [0.7405, 0.2595] │
│ moisture : high rain probability : moderate   │ moisture : high rain probability : moderate  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ harvest [SEP]                                 │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.4907, 0.5093] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.2483, 0.7517] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : moderate water  │ temperature : high humidity : moderate water │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ irrigate [SEP]                               │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : maize stage : vegetative soil    │ [CLS] crop : maize stage : vegetative soil   │ [0.4813, 0.5187] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : moderate humidity : high water  │ temperature : moderate humidity : high water │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ reduce _ irrigation [SEP]                    │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ [CLS] crop : wheat stage : harvesting soil    │ [CLS] crop : wheat stage : harvesting soil   │ [0.7685, 0.2315] │
│ moisture : high rain probability : moderate   │ moisture : high rain probability : moderate  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ harvest [SEP]                                 │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.5166, 0.4834] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : low water       │ temperature : high humidity : low water      │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ improve _ drainage [SEP]                     │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : tomato stage : seedling soil     │ [CLS] crop : tomato stage : seedling soil    │ [0.2291, 0.7709] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : high humidity : moderate water  │ temperature : high humidity : moderate water │                  │
│ availability : adequate which action is more  │ availability : adequate which action is more │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ irrigate [SEP]                               │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ [CLS] crop : maize stage : vegetative soil    │ [CLS] crop : maize stage : vegetative soil   │ [0.5868, 0.4132] │
│ moisture : moderate rain probability : high   │ moisture : moderate rain probability : high  │                  │
│ temperature : moderate humidity : high water  │ temperature : moderate humidity : high water │                  │
│ availability : limited which action is more   │ availability : limited which action is more  │                  │
│ appropriate for this farm state? action :     │ appropriate for this farm state? action :    │                  │
│ wait _ for _ rain [SEP]                       │ reduce _ irrigation [SEP]                    │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=296, training_loss=0.6007950024024861, metrics={'train_runtime': 822.6774, 'train_samples_per_second': 11.548, 'train_steps_per_second': 0.36, 'total_flos': 0.0, 'train_loss': 0.6007950024024861, 'epoch': 1.9932659932659933})

In [27]:
trainer.save_model(output_dir)

---

# <b>Inference

## Loading the saved PEFT model

In [28]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

In [29]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels = 1
)

model = PeftModel.from_pretrained(
    base_model,
    output_dir
)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model.eval()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): MultiHeadSelfAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768

### Using the model for scoring purposes.

In [30]:
prompt = """
Crop: cotton
Stage: flowering
Soil moisture: low
Rain probability: low
Temperature: high
Humidity: low
Water availability: adequate
Which action is more appropriate?
"""

response1 = "WAIT_FOR_RAIN"

response2 = "IRRIGATE"

responses = [response1, response2]

scores = []

for i, response in enumerate(responses, 1):

    text = f"{prompt}\nAction: {response}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        score = model(**inputs).logits.item()

    scores.append(score)

    print(f"Action {i}: {response}")
    print(f"Reward Score: {score:.4f}\n")

best = scores.index(max(scores))

print(f"Best Action: {responses[best]}")

Action 1: WAIT_FOR_RAIN
Reward Score: -0.6082

Action 2: IRRIGATE
Reward Score: 0.8681

Best Action: IRRIGATE
